# 와이즈에프엔(WiseIndex)에서 제공하는 데이터 

- 지수 전체의 성격 파악 (info): 현재 내가 보고 있는 데이터가 어떤 지수의 몇 월 며칠 자 데이터인지 확인합니다.
- 포트폴리오 비중 분석 (list): 실제 어떤 종목이 가장 영향력이 큰지 정렬(Sort)하여 분석할 때 사용합니다.
- 시장 주도 섹터 확인 (sector): 개별 종목보다는 '2차전지', '반도체' 등 특정 산업군에 돈이 몰리는지 확인할 때 유용합니다.
- 장세 판단 (size): 현재 장세가 대형주 위주의 장인지, 아니면 중소형주가 활발하게 움직이는 장인지 비교 분석할 수 있습니다.

| 키 (Key) | 설명 | 포함 데이터 및 용도 |
| :--- | :--- | :--- |
| **info** | 지수 기본 정보 | 지수명, 산출 기관, 기준일자 등 메타데이터가 담겨 있습니다. |
| **list** | 종목별 상세 리스트 | 해당 지수를 구성하는 개별 종목들의 시총, 비중, 등락률 배열입니다. |
| **sector** | 섹터별 요약 정보 | WICS 산업 분류에 따른 섹터별 비중 및 등락 현황을 보여줍니다. |
| **size** | 규모별 요약 정보 | 대형주, 중형주, 소형주 등 기업 크기에 따른 비중 통계입니다. |

In [1]:
from utils import get_yesterday 
from wics import wics_json 

In [11]:
data = get_yesterday()
wics_code = 1010 
result = wics_json(data, wics_code) 

## info
- TRD_AMT (거래대금)의 중요성: 주가가 올라도 거래대금이 터지지 않으면 '가짜 반등'일 확률이 높습니다. 반대로 MKT_VAL은 정체되어 있는데 TRD_AMT가 급증한다면 조만간 큰 변동성이 올 신호로 해석하곤 합니다.
- MKT_VAL vs CNT: 이 둘을 활용하면 해당 날짜의 **종가(Close Price)**를 역산할 수 있습니다.
$$
\text{종가} = \frac{\text{MKT\_VAL}}{\text{CNT}}
$$
- 시계열 분석: TRD_DT를 기준으로 이 데이터들을 정렬하면, 특정 섹터나 종목에 돈이 몰리는지(수급) 아니면 빠져나가는지를 추적할 수 있습니다.

| 컬럼명 (Key) | 설명 | 상세 내용 |
| :--- | :--- | :--- |
| **TRD_DT** | 거래 일자 | 데이터가 생성된 기준 날짜 (예: 20260324) |
| **MKT_VAL** | 시가총액 | 해당 날짜 종가 기준의 기업 가치 총액 |
| **TRD_AMT** | 거래대금 | 해당 거래일에 실제로 사고팔린 금액의 총합 |
| **CNT** | 주식 수 | 지수 산출 또는 시가총액 계산에 반영된 상장 주식 수 |

In [12]:
result['info'].keys()

dict_keys(['TRD_DT', 'MKT_VAL', 'TRD_AMT', 'CNT'])

In [13]:
result['info']

{'TRD_DT': '/Date(1774537200000)/',
 'MKT_VAL': 39460833,
 'TRD_AMT': 870675,
 'CNT': 39}

## sector
- 시장 주도주 파악:
  - IDX_RATE가 가장 높은 섹터가 현재 시장의 방향성을 결정하는 주도 섹터입니다. 예를 들어 IT 섹터의 IDX_RATE가 30%라면, IT 업종이 하락할 때 지수 전체가 방어되기 어렵다는 것을 의미합니다.

- SEC_RATE와 IDX_RATE의 차이:
  - SEC_RATE: "섹터들끼리만 비교했을 때 누가 더 큰가?"를 나타냅니다.
  - IDX_RATE: "지수 전체(100%)에서 이 섹터의 실질적인 영향력은 얼마인가?"를 나타냅니다.

- 데이터 활용 예시:
  - 만약 특정 날짜에 SEC_NM_KOR가 '에너지'인 항목의 IDX_RATE가 평소보다 높아졌다면, 시장 자금이 에너지 관련 종목으로 이동하고 있다고 해석할 수 있습니다.

| 컬럼명 (Key) | 설명 | 상세 의미 및 활용 |
| :--- | :--- | :--- |
| **SEC_CD** | 섹터 코드 | WICS(에너지, 소재 등) 또는 자체 분류 기준에 따른 섹터 고유 코드 |
| **SEC_NM_KOR** | 섹터명 | 해당 섹터의 한글 명칭 (예: IT, 금융, 필수소비재 등) |
| **SEC_RATE** | 섹터 내 비중 (%) | 해당 섹터 그룹 합계를 100%로 보았을 때의 개별 비중 |
| **IDX_RATE** | 지수 내 비중 (%) | 전체 지수(Total Index) 시가총액에서 해당 섹터가 차지하는 비율 |

In [14]:
result['sector'][0].keys()

dict_keys(['SEC_CD', 'SEC_NM_KOR', 'SEC_RATE', 'IDX_RATE'])

In [15]:
result['sector']

[{'SEC_CD': 'G25', 'SEC_NM_KOR': '경기관련소비재', 'SEC_RATE': 7.41, 'IDX_RATE': 0},
 {'SEC_CD': 'G35', 'SEC_NM_KOR': '건강관리', 'SEC_RATE': 7.92, 'IDX_RATE': 0},
 {'SEC_CD': 'G50', 'SEC_NM_KOR': '커뮤니케이션서비스', 'SEC_RATE': 3.28, 'IDX_RATE': 0},
 {'SEC_CD': 'G40', 'SEC_NM_KOR': '금융', 'SEC_RATE': 9.01, 'IDX_RATE': 0},
 {'SEC_CD': 'G10', 'SEC_NM_KOR': '에너지', 'SEC_RATE': 1.75, 'IDX_RATE': 100.0},
 {'SEC_CD': 'G20', 'SEC_NM_KOR': '산업재', 'SEC_RATE': 18.71, 'IDX_RATE': 0},
 {'SEC_CD': 'G55', 'SEC_NM_KOR': '유틸리티', 'SEC_RATE': 0.95, 'IDX_RATE': 0},
 {'SEC_CD': 'G30', 'SEC_NM_KOR': '필수소비재', 'SEC_RATE': 1.34, 'IDX_RATE': 0},
 {'SEC_CD': 'G15', 'SEC_NM_KOR': '소재', 'SEC_RATE': 4.12, 'IDX_RATE': 0},
 {'SEC_CD': 'G45', 'SEC_NM_KOR': 'IT', 'SEC_RATE': 45.51, 'IDX_RATE': 0}]

## size
이 데이터를 통해 현재 시장이 **'어떤 체급'** 의 종목 중심으로 움직이는지 알 수 있습니다.

- 장세의 성격 판단:
  - 대형주 IDX_RATE 상승: 시장의 주도권이 삼성전자, SK하이닉스 같은 우량주에 있으며 지수 위주의 장세임을 뜻합니다.
  - 중/소형주 IDX_RATE 상승: 개별 종목 장세나 테마주 중심의 활발한 거래가 일어나고 있음을 시사합니다.

- SEC_RATE vs IDX_RATE 차이:
  - size 그룹 내에서 **대형주의 SEC_RATE**는 보통 매우 높게 나타납니다(70~80% 이상).
  - 만약 소형주의 IDX_RATE가 SEC_RATE보다 유난히 높게 잡혀 있다면, 해당 지수는 중소형주의 변동성을 더 민감하게 반영하도록 설계된 지수라고 해석할 수 있습니다.

- 투자 전략 활용:
  - 기관이나 외인 자금은 주로 대형주(WMI510)의 IDX_RATE 변화에 민감하게 반응하므로, 이 수치의 추이를 통해 메이저 수급의 방향을 예측해 볼 수 있습니다.

| 컬럼명 (Key) | 설명 | 상세 의미 및 활용 |
| :--- | :--- | :--- |
| **SEC_CD** | 규모 분류 코드 | 기업 크기별 고유 코드 (예: WMI510-대형주, WMI520-중형주 등) |
| **SEC_NM_KOR** | 규모 명칭 | 해당 그룹의 한글 명칭 (예: 대형주, 중형주, 소형주) |
| **SEC_RATE** | 그룹 내 비중 (%) | '규모별 분류' 전체 합(100%) 중 해당 그룹이 차지하는 비율 |
| **IDX_RATE** | 지수 내 비중 (%) | 전체 지수(Total Index) 시가총액에서 해당 규모 그룹이 차지하는 실질적 비율 |

In [16]:
result['size'][0].keys()

dict_keys(['SEC_CD', 'SEC_NM_KOR', 'SEC_RATE', 'IDX_RATE'])

In [17]:
result['size']

[{'SEC_CD': 'WMI510',
  'SEC_NM_KOR': 'WMI500 대형주',
  'SEC_RATE': 78.28,
  'IDX_RATE': 75.4},
 {'SEC_CD': 'WMI520',
  'SEC_NM_KOR': 'WMI500 중형주',
  'SEC_RATE': 11.38,
  'IDX_RATE': 10.62},
 {'SEC_CD': 'WMI530',
  'SEC_NM_KOR': 'WMI500 소형주',
  'SEC_RATE': 10.34,
  'IDX_RATE': 13.97}]

## list
- WGT vs S_WGT vs CAL_WGT:
  - WGT는 결과적인 비중입니다.
  - S_WGT는 대주주 지분 등을 제외하고 실제 시장에서 유통되는 주식(Free-float)만을 고려한 비중입니다.
  - CAL_WGT는 특정 종목의 비중이 너무 커지지 않도록 캡(Cap)을 씌우는 등 지수 산출 기관의 공식이 적용된 비중입니다.

- APT_SHR_CNT (적용 주식수):
  - 단순 상장 주식수가 아니라, 지수 산출 시 가중치를 두기 위해 조정된 주식수입니다. 시가총액을 계산할 때 이 값에 현재가를 곱하여 지수 내 영향력을 결정합니다.

- TOP60:
  - 특정 지수(예: KOSPI 200 등)의 구성 종목을 선정할 때 예비 종목이나 핵심 종목군을 분류하기 위한 관리용 플래그로 활용됩니다.

| 컬럼명 (Key) | 설명 | 상세 내용 및 비고 |
| :--- | :--- | :--- |
| **IDX_CD** | 지수 코드 | 조회 중인 지수의 고유 코드 (예: WMI500) |
| **IDX_NM_KOR** | 지수명 | 지수의 한글 명칭 (예: WMI500 지수) |
| **ALL_MKT_VAL** | 전체 시가총액 | 해당 지수에 포함된 모든 종목의 시가총액 합계 |
| **CMP_CD** | 종목 코드 | 개별 상장 종목의 6자리 단축 코드 |
| **CMP_KOR** | 종목명 | 개별 상장 종목의 한글 명칭 |
| **MKT_VAL** | 종목 시가총액 | 해당 종목의 현재 시가총액 |
| **WGT** | 비중 (Weight) | 지수 내에서 해당 종목이 차지하는 최종 비중 (%) |
| **S_WGT** | 유동 비중 | 실제 시장에서 거래 가능한 주식 비율을 반영한 비중 |
| **CAL_WGT** | 산출 비중 | 지수 계산 시 적용되는 계산용 가중치 |
| **SEC_CD** | 섹터 코드 | 해당 종목이 속한 WICS 또는 규모별 섹터 코드 |
| **SEC_NM_KOR** | 섹터명 | 해당 종목이 속한 섹터의 한글 명칭 |
| **SEQ** | 순번 (Sequence) | 지수 내 비중 순위 또는 정렬 순서 |
| **TOP60** | 상위 60 여부 | 지수 내 상위 60위 이내 포함 여부 (보통 1 또는 0) |
| **APT_SHR_CNT** | 적용 주식수 | 지수 산출에 실제 반영되는 (유동성 등이 고려된) 주식수 |

In [18]:
result['list'][0].keys()

dict_keys(['IDX_CD', 'IDX_NM_KOR', 'ALL_MKT_VAL', 'CMP_CD', 'CMP_KOR', 'MKT_VAL', 'WGT', 'S_WGT', 'CAL_WGT', 'SEC_CD', 'SEC_NM_KOR', 'SEQ', 'TOP60', 'APT_SHR_CNT'])

In [19]:
result['list'][0]

{'IDX_CD': 'G1010',
 'IDX_NM_KOR': 'WICS 에너지',
 'ALL_MKT_VAL': 39460833,
 'CMP_CD': '034730',
 'CMP_KOR': 'SK',
 'MKT_VAL': 11139315,
 'WGT': 28.23,
 'S_WGT': 28.23,
 'CAL_WGT': 1.0,
 'SEC_CD': 'G10',
 'SEC_NM_KOR': '에너지',
 'SEQ': 1,
 'TOP60': 4,
 'APT_SHR_CNT': 33351243}